# MedoraAI Brain MRI Training - MobileNetV2

This notebook trains the `.h5` file used by the MedoraAI backend:

```env
BRAIN_MODEL_PATH=./models/brain_tumor_mobilenetv2.h5
```

It follows the existing `brain_tumor_analysis` project:

- Dataset: Kaggle `navoneel/brain-mri-images-for-brain-tumor-detection`
- Classes: `yes` = Tumor, `no` = No Tumor
- Input: 128 x 128 RGB, normalized 0-1
- Architecture: MobileNetV2 -> GlobalAveragePooling2D -> BatchNorm -> Dense -> Dropout -> Sigmoid

The exported model is a Keras `.h5` file compatible with `backend/services/brain_classifier.py`.

In [ ]:
!nvidia-smi || true
# Colab already includes TensorFlow, NumPy, OpenCV, scikit-learn, and matplotlib.
# Do not force-reinstall TensorFlow/NumPy/OpenCV here; it can break binary compatibility.
!pip -q install kagglehub

In [ ]:
import os, random, shutil, json, time
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
import cv2

import tensorflow as tf
from tensorflow.keras import layers, Sequential
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, roc_curve

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

IMG_SIZE = (128, 128)
BATCH_SIZE = 16
EPOCHS_HEAD = 12
EPOCHS_FINE = 8
OUT_PATH = Path('/content/brain_tumor_mobilenetv2.h5')
print(tf.__version__)

In [ ]:
# Download dataset using kagglehub. No kaggle.json is usually needed for public datasets in Colab.
import kagglehub
path = kagglehub.dataset_download('navoneel/brain-mri-images-for-brain-tumor-detection')
DATASET_PATH = Path(path)
print('Dataset:', DATASET_PATH)
print(list(DATASET_PATH.iterdir()))

In [ ]:
# Load images into memory. Dataset is small enough for this.
X, y, paths = [], [], []
label_map = {'no': 0, 'yes': 1}

for label_name, label_value in label_map.items():
    for p in sorted((DATASET_PATH / label_name).glob('*')):
        if p.suffix.lower() not in ['.jpg', '.jpeg', '.png']:
            continue
        img = cv2.imread(str(p))
        if img is None:
            continue
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        img = cv2.resize(img, IMG_SIZE)
        X.append(img.astype('float32') / 255.0)
        y.append(label_value)
        paths.append(str(p))

X = np.array(X, dtype='float32')
y = np.array(y, dtype='int32')
print('X:', X.shape, 'y:', y.shape, 'tumor count:', int(y.sum()), 'no tumor:', int((1-y).sum()))

In [ ]:
X_train, X_tmp, y_train, y_tmp = train_test_split(X, y, test_size=0.30, random_state=SEED, stratify=y)
X_val, X_test, y_val, y_test = train_test_split(X_tmp, y_tmp, test_size=0.50, random_state=SEED, stratify=y_tmp)
print('Train:', X_train.shape, 'Val:', X_val.shape, 'Test:', X_test.shape)

train_aug = tf.keras.preprocessing.image.ImageDataGenerator(
    rotation_range=12,
    width_shift_range=0.06,
    height_shift_range=0.06,
    zoom_range=0.08,
    horizontal_flip=True,
    fill_mode='nearest',
)

In [ ]:
base_model = MobileNetV2(
    input_shape=(128, 128, 3),
    include_top=False,
    weights='imagenet',
)
base_model.trainable = False

model = Sequential([
    base_model,
    layers.GlobalAveragePooling2D(),
    layers.BatchNormalization(),
    layers.Dense(128, activation='relu'),
    layers.Dropout(0.5),
    layers.Dense(1, activation='sigmoid'),
])

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss='binary_crossentropy',
    metrics=['accuracy', tf.keras.metrics.Precision(name='precision'), tf.keras.metrics.Recall(name='recall'), tf.keras.metrics.AUC(name='auc')],
)
model.summary()

In [ ]:
callbacks = [
    EarlyStopping(monitor='val_auc', mode='max', patience=5, restore_best_weights=True),
    ReduceLROnPlateau(monitor='val_auc', mode='max', patience=2, factor=0.3, min_lr=1e-6),
    ModelCheckpoint('/content/brain_tumor_best.keras', monitor='val_auc', mode='max', save_best_only=True),
]

history = model.fit(
    train_aug.flow(X_train, y_train, batch_size=BATCH_SIZE, seed=SEED),
    validation_data=(X_val, y_val),
    epochs=EPOCHS_HEAD,
    callbacks=callbacks,
)

In [ ]:
# Fine tune last MobileNetV2 layers
base_model.trainable = True
for layer in base_model.layers[:-20]:
    layer.trainable = False

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-5),
    loss='binary_crossentropy',
    metrics=['accuracy', tf.keras.metrics.Precision(name='precision'), tf.keras.metrics.Recall(name='recall'), tf.keras.metrics.AUC(name='auc')],
)

history_ft = model.fit(
    train_aug.flow(X_train, y_train, batch_size=BATCH_SIZE, seed=SEED),
    validation_data=(X_val, y_val),
    epochs=EPOCHS_FINE,
    callbacks=callbacks,
)

In [ ]:
# Evaluation
probs = model.predict(X_test).flatten()
preds = (probs >= 0.5).astype('int32')
print(classification_report(y_test, preds, target_names=['No Tumor', 'Tumor']))
print('ROC AUC:', roc_auc_score(y_test, probs))

cm = confusion_matrix(y_test, preds)
fig, ax = plt.subplots(figsize=(4, 4))
im = ax.imshow(cm, cmap='Blues')
ax.set_xticks([0, 1], labels=['No Tumor', 'Tumor'])
ax.set_yticks([0, 1], labels=['No Tumor', 'Tumor'])
ax.set_xlabel('Predicted')
ax.set_ylabel('True')
for i in range(cm.shape[0]):
    for j in range(cm.shape[1]):
        ax.text(j, i, cm[i, j], ha='center', va='center', color='black')
fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
plt.show()

In [ ]:
# Save legacy H5 because backend uses tf.keras.models.load_model(model_path)
model.save(OUT_PATH)

manifest = {
    'backend_expected_model': 'Keras Sequential MobileNetV2 binary sigmoid',
    'input_size': [128, 128, 3],
    'labels': {'0': 'No Tumor', '1': 'Tumor'},
    'threshold': 0.5,
    'model_path': str(OUT_PATH),
}
Path('/content/brain_tumor_mobilenetv2.labels.json').write_text(json.dumps(manifest, indent=2))
print('Saved:', OUT_PATH)
print(Path('/content/brain_tumor_mobilenetv2.labels.json').read_text())

In [ ]:
# Copy to Google Drive for download/persistence
from google.colab import drive
drive.mount('/content/drive')
DEST_DIR = Path('/content/drive/MyDrive/MedoraAI/models')
DEST_DIR.mkdir(parents=True, exist_ok=True)
shutil.copy2(OUT_PATH, DEST_DIR / OUT_PATH.name)
shutil.copy2('/content/brain_tumor_mobilenetv2.labels.json', DEST_DIR / 'brain_tumor_mobilenetv2.labels.json')
print('Saved to:', DEST_DIR)
print('Use locally: BRAIN_MODEL_PATH=./models/brain_tumor_mobilenetv2.h5')

## Local MedoraAI Setup After Download

Put the downloaded model file here in your repo:

```text
MedoraAI/models/brain_tumor_mobilenetv2.h5
```

Then set:

```env
BRAIN_MODEL_PATH=./models/brain_tumor_mobilenetv2.h5
```